# naturaDock Example Workflow

End-to-end virtual screening pipeline using **AutoDock Vina**, **RDKit**, and **Biopython**.

This notebook walks through:
1. Verifying dependencies
2. Preparing input files (protein PDB + ligand SDF)
3. Running the naturaDock pipeline via Docker
4. Analyzing and visualizing results

---

> **Recommended**: Run this pipeline inside Docker for full reproducibility.
> See the README for the Docker run command.

## 1. Check Dependencies

In [ ]:
import sys
import os
import shutil
import subprocess
from pathlib import Path

# Check required packages
required = ['rdkit', 'biopython', 'pandas', 'matplotlib', 'seaborn', 'toml', 'meeko', 'tqdm', 'psutil']
missing = []
for pkg in required:
    try:
        __import__(pkg if pkg != 'biopython' else 'Bio')
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} — MISSING')
        missing.append(pkg)

if missing:
    print(f'\nInstall missing packages: pip install {" ".join(missing)}')
else:
    print('\nAll dependencies satisfied.')

# Check AutoDock Vina
vina = os.environ.get('VINA_EXECUTABLE') or shutil.which('vina') or shutil.which('vina.exe')
if vina:
    print(f'  ✓ AutoDock Vina: {vina}')
else:
    print('  ✗ AutoDock Vina not found — install it or set VINA_EXECUTABLE env var')

## 2. Configure Paths

Edit the paths below to point to your input files and desired output directory.

In [ ]:
from pathlib import Path

# --- EDIT THESE ---
PROTEIN_PDB   = Path("protein.pdb")       # Path to your protein PDB file
LIGANDS_SDF   = Path("ligands.sdf")       # Path to your compound library SDF
OUTPUT_DIR    = Path("output")            # Output directory

# Docking box (Angstroms) — tighten around the binding site for real runs
BOX_SIZE_X, BOX_SIZE_Y, BOX_SIZE_Z = 22.0, 22.0, 22.0

# Lipinski-inspired filters (relax for large natural products)
MAX_MOL_WEIGHT     = 700.0
MAX_ROTATABLE_BONDS = 12
MIN_LOGP, MAX_LOGP  = -5.0, 6.0
# ------------------

OUTPUT_DIR.mkdir(exist_ok=True)

# Validate inputs exist
for f in [PROTEIN_PDB, LIGANDS_SDF]:
    status = '✓' if f.exists() else '✗ NOT FOUND'
    print(f'  {status}  {f}')

## 3. Quick Protein Inspection

Validates the PDB and reports any structural issues before docking.

In [ ]:
from Bio.PDB import PDBParser
from pdbfixer import PDBFixer

parser = PDBParser(QUIET=True)
structure = parser.get_structure(PROTEIN_PDB.stem, str(PROTEIN_PDB))

chains   = list(structure.get_chains())
residues = list(structure.get_residues())
atoms    = list(structure.get_atoms())

print(f'Protein: {PROTEIN_PDB.name}')
print(f'  Chains:   {len(chains)}  ({", ".join(c.id for c in chains)})')
print(f'  Residues: {len(residues)}')
print(f'  Atoms:    {len(atoms)}')

# PDBFixer validation
fixer = PDBFixer(str(PROTEIN_PDB))
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.findMissingAtoms()

print(f'\nValidation:')
print(f'  Missing residues:      {len(fixer.missingResidues)}')
print(f'  Non-standard residues: {len(fixer.nonstandardResidues)}')
print(f'  Missing atoms:         {len(fixer.missingAtoms)}')

if fixer.nonstandardResidues:
    print(f'  Non-standard: {fixer.nonstandardResidues}')

## 4. Compound Library Inspection

Loads compounds and previews their properties before filtering.

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

supplier = Chem.SDMolSupplier(str(LIGANDS_SDF))
mols = [m for m in supplier if m is not None]

print(f'Loaded {len(mols)} valid molecules from {LIGANDS_SDF.name}')

rows = []
for mol in mols:
    name = mol.GetProp('_Name') if mol.HasProp('_Name') else 'unnamed'
    rows.append({
        'name':             name,
        'mol_weight':       round(Descriptors.MolWt(mol), 2),
        'rotatable_bonds':  Descriptors.NumRotatableBonds(mol),
        'logP':             round(Descriptors.MolLogP(mol), 2),
        'HBD':              Descriptors.NumHDonors(mol),
        'HBA':              Descriptors.NumHAcceptors(mol),
    })

df_compounds = pd.DataFrame(rows)

# Flag compounds that would pass the filter
df_compounds['passes_filter'] = (
    (df_compounds['mol_weight']      <= MAX_MOL_WEIGHT) &
    (df_compounds['rotatable_bonds'] <= MAX_ROTATABLE_BONDS) &
    (df_compounds['logP']            >= MIN_LOGP) &
    (df_compounds['logP']            <= MAX_LOGP)
)

print(f'Compounds passing filter: {df_compounds["passes_filter"].sum()} / {len(df_compounds)}')
df_compounds

## 5. Run the naturaDock Pipeline

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Find project root (contains src/)
def find_project_root(start):
    for p in [Path(start)] + list(Path(start).parents):
        if (p / 'src').exists() and (p / 'pyproject.toml').exists():
            return p
    return Path(start)

project_root = find_project_root(Path.cwd())
print(f'Project root: {project_root}')

env = os.environ.copy()
env['PYTHONPATH'] = str(project_root / 'src')

cmd = [
    sys.executable, '-m', 'naturaDock.main',
    '--protein',              str(PROTEIN_PDB),
    '--ligands',              str(LIGANDS_SDF),
    '--output',               str(OUTPUT_DIR),
    '--size_x',               str(BOX_SIZE_X),
    '--size_y',               str(BOX_SIZE_Y),
    '--size_z',               str(BOX_SIZE_Z),
    '--max_mol_weight',       str(MAX_MOL_WEIGHT),
    '--max_rotatable_bonds',  str(MAX_ROTATABLE_BONDS),
    '--min_logp',             str(MIN_LOGP),
    '--max_logp',             str(MAX_LOGP),
]

print(f'Running: {" ".join(cmd)}\n')

result = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=str(project_root))
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

if result.returncode == 0:
    print('\n✓ Pipeline completed successfully.')
else:
    print(f'\n✗ Pipeline failed (exit code {result.returncode})')

## 6. Analyze Results

In [ ]:
import pandas as pd

results_csv = OUTPUT_DIR / 'ranked_results.csv'

if not results_csv.exists():
    print('No results found. Check that docking completed successfully.')
else:
    df = pd.read_csv(results_csv)
    print(f'Docked {len(df)} compounds\n')
    print(df.to_string(index=False))
    print(f'\nBest  affinity: {df["affinity"].min():.3f} kcal/mol  ({df.loc[df["affinity"].idxmin(), "compound"]})')
    print(f'Worst affinity: {df["affinity"].max():.3f} kcal/mol  ({df.loc[df["affinity"].idxmax(), "compound"]})')
    print(f'Mean  affinity: {df["affinity"].mean():.3f} kcal/mol')

## 7. Visualize Score Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

results_csv = OUTPUT_DIR / 'ranked_results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('naturaDock — Docking Results', fontsize=14, fontweight='bold')

    # Distribution
    sns.histplot(df['affinity'], kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title('Score Distribution')
    axes[0].set_xlabel('Binding Affinity (kcal/mol)')
    axes[0].set_ylabel('Count')
    axes[0].axvline(df['affinity'].mean(), color='red', linestyle='--', label=f'Mean: {df["affinity"].mean():.2f}')
    axes[0].legend()

    # Ranked bar chart (top 20)
    top = df.head(20).sort_values('affinity')
    axes[1].barh(top['compound'], top['affinity'], color='steelblue')
    axes[1].set_title('Top 20 Compounds by Affinity')
    axes[1].set_xlabel('Binding Affinity (kcal/mol)')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'docking_summary.png', dpi=150)
    plt.show()
    print(f'Plot saved to {OUTPUT_DIR}/docking_summary.png')
else:
    print('No results to plot.')

## 8. Statistical Summary

In [ ]:
summary_path = OUTPUT_DIR / 'statistical_summary.txt'

if summary_path.exists():
    print('=== Statistical Summary ===')
    print(summary_path.read_text())

# Also print from pandas for richer stats
results_csv = OUTPUT_DIR / 'ranked_results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    print('\n=== Extended Statistics ===')
    print(df['affinity'].describe().round(3).to_string())

    # Actives threshold: commonly < -7.0 kcal/mol is considered strong binding
    strong = df[df['affinity'] < -7.0]
    print(f'\nStrong binders (< -7.0 kcal/mol): {len(strong)} / {len(df)}')
    if not strong.empty:
        print(strong.to_string(index=False))

## 9. Output Files

In [ ]:
print(f'Output directory: {OUTPUT_DIR.resolve()}\n')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        size_str = f'{size/1024:.1f} KB' if size > 1024 else f'{size} B'
        rel = f.relative_to(OUTPUT_DIR)
        print(f'  {str(rel):<50} {size_str}')